# 2. Supervised Modeling

En esta sección integramos el preprocesamiento modular del proyecto con varios clasificadores base de Scikit-Learn y evaluamos su desempeño con validación cruzada estratificada, priorizando Recall y F1 por el fuerte desbalance del dataset.

In [1]:
# ==========================================
# 1. IMPORTS Y CONFIGURACION
# ==========================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn import set_config

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    parent_root = PROJECT_ROOT.parent
    if (parent_root / 'src').exists():
        PROJECT_ROOT = parent_root
    else:
        candidate = PROJECT_ROOT / 'mi_proyecto'
        if (candidate / 'src').exists():
            PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import UnknownToNaN, SmartImputer, OutlierCapper
from src.model_training import get_model_registry
from src.model_evaluation import build_stratified_kfold, print_classification_cv_report, print_model_comparison_report

set_config(display='diagram')

print(f'Proyecto detectado en: {PROJECT_ROOT}')
print('Imports sincronizados con src/ correctamente')


Proyecto detectado en: /home/tomy/Downloads/cdd_ev2/mi_proyecto
Imports sincronizados con src/ correctamente


In [2]:
# ==========================================
# 2. CARGA DE DATOS
# ==========================================

data_path = PROJECT_ROOT / 'data' / 'raw' / 'healthcare-dataset-stroke-data.csv'
df_raw = pd.read_csv(data_path)

print(f'Dataset cargado desde: {data_path}')
print(f'Dimensiones: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas')
df_raw.head()

Dataset cargado desde: /home/tomy/Downloads/cdd_ev2/mi_proyecto/data/raw/healthcare-dataset-stroke-data.csv
Dimensiones: 5110 filas, 12 columnas


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
# ==========================================
# 3. DEFINICION DE VARIABLES
# ==========================================

target = 'stroke'
X_raw = df_raw.drop(columns=[target, 'id'])
y_raw = df_raw[target]

numeric_features = X_raw.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()
categorical_features = X_raw.select_dtypes(include=['object', 'string', 'category', 'bool']).columns.tolist()

print(f'Features numéricas ({len(numeric_features)}): {numeric_features}')
print(f'Features categóricas ({len(categorical_features)}): {categorical_features}')
print('\nDistribución del target:')
print(y_raw.value_counts(normalize=True).mul(100).round(2).to_string())


Features numéricas (5): ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
Features categóricas (5): ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

Distribución del target:
stroke
0    95.13
1     4.87


In [4]:
# ==========================================
# 4. PREPROCESAMIENTO BASE PARA LOS MODELOS
# ==========================================

feature_preprocessor = Pipeline([
    ('unknown_to_nan', UnknownToNaN(columns=categorical_features)),
    ('smart_imputer', SmartImputer()),
    ('outlier_capper', OutlierCapper(columns=numeric_features)),
    ('feature_encoding', ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('scaler', StandardScaler())
            ]), numeric_features),
            ('cat', Pipeline([
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), categorical_features),
        ],
        remainder='drop'
    ))
])

print('Preprocesador supervisado configurado con UnknownToNaN, SmartImputer, OutlierCapper, StandardScaler y OneHotEncoder')

Preprocesador supervisado configurado con UnknownToNaN, SmartImputer, OutlierCapper, StandardScaler y OneHotEncoder


In [5]:
# ==========================================
# 5. PIPELINES DE MODELO + PREPROCESAMIENTO
# ==========================================

model_registry = get_model_registry(random_state=42)

def build_model_pipeline(estimator):
    """Construye un Pipeline completo con preprocesamiento y clasificador."""
    return Pipeline([
        ('preprocessing', feature_preprocessor),
        ('classifier', estimator),
    ])

model_pipelines = {
    name: build_model_pipeline(estimator)
    for name, estimator in model_registry.items()
}

print('Modelos base disponibles:')
for name in model_pipelines:
    print(f'- {name}')

Modelos base disponibles:
- logistic_regression
- random_forest
- svc


In [6]:
# ==========================================
# 6. EVALUACION BASE CON VALIDACION CRUZADA
# ==========================================

cv = build_stratified_kfold(n_splits=5, random_state=42)
prioritized_report = print_model_comparison_report(
    models=model_pipelines,
    X=X_raw,
    y=y_raw,
    cv=cv,
    random_state=42,
)

prioritized_report = prioritized_report.sort_values(
    by=['recall_mean', 'f1_mean', 'roc_auc_mean', 'precision_mean'],
    ascending=[False, False, False, False],
).reset_index(drop=True)

prioritized_report



=== Model Comparison ===
              model  precision_mean  precision_std  recall_mean  recall_std  f1_mean   f1_std  roc_auc_mean  roc_auc_std
logistic_regression        0.130771       0.009227     0.802939    0.060460 0.224889 0.015831      0.834207     0.024556
                svc        0.121665       0.015607     0.702367    0.075616 0.207377 0.025883      0.798383     0.025469
      random_forest        0.198333       0.142449     0.020082    0.014143 0.036137 0.025286      0.771874     0.034032


,model,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,logistic_regression,0.130771,0.009227,0.802939,0.060460,0.224889,0.015831,0.834207,0.024556
1,svc,0.121665,0.015607,0.702367,0.075616,0.207377,0.025883,0.798383,0.025469
2,random_forest,0.198333,0.142449,0.020082,0.014143,0.036137,0.025286,0.771874,0.034032


In [7]:
# ==========================================
# 7. REPORTE DETALLADO DEL MEJOR CANDIDATO
# ==========================================

best_model_name = prioritized_report.iloc[0]['model']
best_model_pipeline = model_pipelines[best_model_name]

print(f'\nMejor candidato según Recall/F1: {best_model_name}')
best_fold_report = print_classification_cv_report(
    model_name=best_model_name,
    estimator=best_model_pipeline,
    X=X_raw,
    y=y_raw,
    cv=cv,
    random_state=42,
)
best_fold_report



Mejor candidato según Recall/F1: logistic_regression

=== logistic_regression ===
Cross-validated metrics (mean ± std)
Precision: 0.1308 ± 0.0092
Recall:    0.8029 ± 0.0605
F1-score:  0.2249 ± 0.0158
ROC-AUC:   0.8342 ± 0.0246

Fold-level detail:
 fold  precision   recall       f1  roc_auc
    1   0.126280 0.740000 0.215743 0.822181
    2   0.131250 0.840000 0.227027 0.831317
    3   0.138614 0.840000 0.237960 0.858189
    4   0.140065 0.860000 0.240896 0.858333
    5   0.117647 0.734694 0.202817 0.801015


,fold,precision,recall,f1,roc_auc
0,1,0.126280,0.740000,0.215743,0.822181
1,2,0.131250,0.840000,0.227027,0.831317
2,3,0.138614,0.840000,0.237960,0.858189
3,4,0.140065,0.860000,0.240896,0.858333
4,5,0.117647,0.734694,0.202817,0.801015
